# 02 — Preprocessing & Feature Engineering

Prepares the Ames Housing dataset for modeling.  
Covers: train/val/test split, missing value handling (structural, MAR, MCAR), feature engineering, and categorical grouping.

**Scope:** Preprocessing only. No EDA plots or modeling.  
**Output:** Processed CSVs saved to `../data/processed/`.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 2. Data Loading

In [2]:
data_path = '../data/raw/AmesHousing.csv'
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')

Dataset shape: (2930, 82)


## 3. Train / Validation / Test Split

In [3]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=40
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (2051, 81) (2051,)
Validation: (439, 81) (439,)
Test: (440, 81) (440,)


In [4]:
train_model = df.loc[X_train.index].copy()
val_model   = df.loc[X_val.index].copy()
test_model  = df.loc[X_test.index].copy()

## 4. Missing Value Handling — Structural

Variables where NA reflects absence of the feature (e.g., no garage, no basement).  
Categoricals → `'None'`; numerics → `0`.

In [13]:
structural_categorical = [
    'Alley',
    'Mas Vnr Type',
    'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
    'BsmtFin Type 1', 'BsmtFin Type 2',
    'Fireplace Qu',
    'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
    'Pool QC', 'Fence', 'Misc Feature'
]

structural_numeric = [
    'Mas Vnr Area',
    'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
    'Bsmt Full Bath', 'Bsmt Half Bath',
    'Garage Yr Blt', 'Garage Cars', 'Garage Area'
]

for df_ in [train_model, val_model, test_model]:
    for col in structural_categorical:
        if col in df_.columns:
            df_[col] = df_[col].fillna('None')
    for col in structural_numeric:
        if col in df_.columns:
            df_[col] = df_[col].fillna(0)

print("Structural missing values filled.")

Structural missing values filled.


## 5. Missing Value Handling — Non-Structural

- **Lot Frontage**: MAR → impute by neighborhood median (calculated on train only).
- **Electrical**: 1 missing → assume MCAR, impute with train mode.

In [6]:
# Lot Frontage: MAR → impute by neighborhood median
lot_frontage_medians = train_model.groupby('Neighborhood')['Lot Frontage'].median()
global_lot_frontage_median = train_model['Lot Frontage'].median()


def impute_lot_frontage(df, medians, global_median):
    df = df.copy()
    missing_idx = df['Lot Frontage'].isna()
    df.loc[missing_idx, 'Lot Frontage'] = (
        df.loc[missing_idx, 'Neighborhood']
        .map(medians)
        .fillna(global_median)
    )
    return df


train_model = impute_lot_frontage(train_model, lot_frontage_medians, global_lot_frontage_median)
val_model   = impute_lot_frontage(val_model,   lot_frontage_medians, global_lot_frontage_median)
test_model  = impute_lot_frontage(test_model,  lot_frontage_medians, global_lot_frontage_median)

# Electrical: MCAR → impute with train mode
electrical_mode = train_model['Electrical'].mode()[0]

for df_ in [train_model, val_model, test_model]:
    if 'Electrical' in df_.columns:
        df_['Electrical'] = df_['Electrical'].fillna(electrical_mode)

print("Non-structural missing values filled.")

Non-structural missing values filled.


## 6. Feature Engineering

Derived features based on domain knowledge. Definitions are applied identically across all splits.

In [7]:
for df_ in [train_model, val_model, test_model]:
    # Garage existence indicator (derived from structural imputation)
    df_['Has_Garage'] = (df_['Garage Area'] > 0).astype(int)

    # Total living area
    df_['Total_SF'] = df_['Total Bsmt SF'] + df_['Gr Liv Area']

    # Age at time of sale
    df_['House_Age'] = df_['Yr Sold'] - df_['Year Built']

    # Years since last remodel
    df_['Years_Since_Remod'] = df_['Yr Sold'] - df_['Year Remod/Add']

    # Total bathrooms (full + half weighted)
    df_['Total_Bath'] = (
        df_['Full Bath'] + 0.5 * df_['Half Bath'] +
        df_['Bsmt Full Bath'] + 0.5 * df_['Bsmt Half Bath']
    )

    # Quality × Condition interaction
    df_['Qual_Cond'] = df_['Overall Qual'] * df_['Overall Cond']

    # Log-transformed target (right-skewed)
    df_['log_SalePrice'] = np.log(df_['SalePrice'])

print("Feature engineering complete.")
train_model[['Total_SF', 'House_Age', 'Years_Since_Remod', 'Total_Bath', 'Qual_Cond', 'log_SalePrice']].describe()

Feature engineering complete.


,Total_SF,House_Age,Years_Since_Remod,Total_Bath,Qual_Cond,log_SalePrice
count,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000
mean,2556.921989,36.232082,23.346173,2.225987,33.891760,12.026308
std,832.564179,30.413825,20.936352,0.813905,9.089151,0.405545
min,612.000000,-1.000000,-2.000000,1.000000,1.000000,9.480368
25%,2004.000000,7.000000,4.000000,2.000000,30.000000,11.774905
50%,2446.000000,34.000000,15.000000,2.000000,35.000000,11.995352
75%,2999.500000,54.000000,42.000000,2.500000,40.000000,12.271977
max,11752.000000,136.000000,60.000000,7.000000,90.000000,13.534473


## 7. Categorical Grouping

Rare-level consolidation and ordinal simplification. All grouping rules are fitted on the train set only.

In [8]:
def group_rare_from_train(train_series, other_series_list=None, threshold=0.05):
    """Group rare levels (< threshold) into 'Other'. Fitted on train only."""
    freq = train_series.value_counts(normalize=True)
    rare_levels = freq[freq < threshold].index.tolist()

    train_grouped = train_series.apply(lambda x: 'Other' if x in rare_levels else x)
    grouped_others = [
        s.apply(lambda x: 'Other' if x in rare_levels else x)
        for s in (other_series_list or [])
    ]
    return train_grouped, grouped_others, rare_levels


# Neighborhood
train_model['Neighborhood_grouped'], grouped_sets, _ = group_rare_from_train(
    train_model['Neighborhood'],
    [val_model['Neighborhood'], test_model['Neighborhood']],
    threshold=0.05
)
val_model['Neighborhood_grouped']  = grouped_sets[0]
test_model['Neighborhood_grouped'] = grouped_sets[1]


# Neighborhood simplified (high-value vs other)
def simplify_neighborhood(series):
    high_value = ['NridgHt', 'Somerst']
    return series.apply(lambda x: 'High' if x in high_value else 'Other')

for df_ in [train_model, val_model, test_model]:
    df_['Neighborhood_simple'] = simplify_neighborhood(df_['Neighborhood_grouped'])


# Kitchen Quality grouped (ordinal collapse)
def group_kitchen_qual(series):
    mapping = {'Po': 'Low', 'Fa': 'Low', 'TA': 'Medium', 'Gd': 'High', 'Ex': 'High'}
    return series.map(mapping).fillna(series)

for df_ in [train_model, val_model, test_model]:
    df_['Kitchen_Qual_grouped'] = group_kitchen_qual(df_['Kitchen Qual'])


# Building Type simplified
def simplify_bldg_type(series):
    return series.apply(lambda x: 'Detached' if x == '1Fam' else 'Other')

for df_ in [train_model, val_model, test_model]:
    df_['Bldg_Type_simple'] = simplify_bldg_type(df_['Bldg Type'])

print("Categorical grouping complete.")

Categorical grouping complete.


## 8. Price Tier Target Variable (Classification)

Tier thresholds are computed from the train set only.

In [9]:
q1, q2 = train_model['SalePrice'].quantile([1/3, 2/3])


def assign_price_tier(price, q1, q2):
    if price <= q1: return 'Low'
    elif price <= q2: return 'Medium'
    return 'High'


for df_ in [train_model, val_model, test_model]:
    df_['price_tier'] = df_['SalePrice'].apply(lambda x: assign_price_tier(x, q1, q2))
    df_['is_high'] = (df_['price_tier'] == 'High').astype(int)

print("Price tier distribution (train):")
print(train_model['price_tier'].value_counts())
print("\nis_high distribution (train):")
print(train_model['is_high'].value_counts(normalize=True))

Price tier distribution (train):
price_tier
Low       705
High      676
Medium    670
Name: count, dtype: int64

is_high distribution (train):
is_high
0    0.670405
1    0.329595
Name: proportion, dtype: float64


## 9. Shape Checks and Missing Value Verification

In [10]:
print("=== Dataset shapes ===")
print(f"Train:      {train_model.shape}")
print(f"Validation: {val_model.shape}")
print(f"Test:       {test_model.shape}")

=== Dataset shapes ===
Train:      (2051, 95)
Validation: (439, 95)
Test:       (440, 95)


In [11]:
# Columns used in modeling
modeling_features = [
    'Overall Qual', 'Total_SF', 'Lot Area', 'Garage Area', 'Total_Bath',
    'House_Age', 'Neighborhood_grouped', 'Neighborhood_simple',
    'Kitchen_Qual_grouped', 'Bldg_Type_simple',
    'log_SalePrice', 'SalePrice', 'is_high', 'price_tier'
]

print("=== Missing values in modeling features ===")
for split_name, df_ in [('Train', train_model), ('Val', val_model), ('Test', test_model)]:
    missing = df_[modeling_features].isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"{split_name}: ✓ No missing values")
    else:
        print(f"{split_name}: ✗ Missing values found:\n{missing}")

=== Missing values in modeling features ===
Train: ✓ No missing values
Val: ✓ No missing values
Test: ✓ No missing values


## 10. Export Processed Datasets

In [12]:
import os

os.makedirs('../data/processed', exist_ok=True)

train_model.to_csv('../data/processed/train.csv', index=False)
val_model.to_csv('../data/processed/val.csv', index=False)
test_model.to_csv('../data/processed/test.csv', index=False)

print("Processed datasets saved to ../data/processed/")
print(f"  train.csv  — {train_model.shape}")
print(f"  val.csv    — {val_model.shape}")
print(f"  test.csv   — {test_model.shape}")

Processed datasets saved to ../data/processed/
  train.csv  — (2051, 95)
  val.csv    — (439, 95)
  test.csv   — (440, 95)
